<a href="https://colab.research.google.com/github/robert13613/LCF/blob/stenardu-patch-1/estrazione_messaggi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:

!pip install -q python-docx

# Monta Google Drive per mantenere i file al sicuro sul tuo cloud personale
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import csv
import re
from docx import Document

# Riconosce l'inizio di un nuovo messaggio (es. EMAIL 01, WHATSAPP 02, NOTA 01, RECENSIONE 01)
HEADER_FONTE_RE = re.compile(
    r"^(?:Fonte\s*:\s*.+|(?:EMAIL|WHATSAPP|MESSAGGIO|NOTA|NOTE|RECENSIONE|POST)(?:\s+\d+)?)$"
)

# Riconosce i titoli di sezione del documento (es. "1. Email di donatori e partner")
TITOLO_SEZIONE_RE = re.compile(r"^\d+\.\s+[A-ZÀ-Ü]")

INTESTAZIONI_AUTORE = ("da:", "autore:", "mittente:", "operatore:", "utente:")
INTESTAZIONI_DATA = ("data:", "inviato il:", "del:")


def extract_messages_to_csv(docx_path: str, csv_path: str) -> int:
    """
    Legge il file Word, estrae una riga per messaggio (fonte, data, autore, testo)
    ignorando introduzione e titoli di sezione, e salva il CSV.
    """
    doc = Document(docx_path)
    messages = []

    current_msg = None
    current_text = []

    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue

        # 1. Se incontra l'intestazione di un nuovo messaggio (es. EMAIL 01)
        if HEADER_FONTE_RE.match(text):
            if current_msg is not None:
                current_msg["testo"] = "\n".join(current_text).strip()
                messages.append(current_msg)

            fonte_val = text.split(":", 1)[-1].strip().upper()
            current_msg = {"fonte": fonte_val}
            current_text = []
            continue

        # Ignora tutto ciò che precede il primo messaggio (titolo e intro del file)
        if current_msg is None:
            continue

        # Ignora i titoli numerati delle sezioni (es. "2. Messaggi dei volontari")
        if TITOLO_SEZIONE_RE.match(text):
            continue

        text_lower = text.lower()

        # 2. Estrazione metadati (solo prima che inizi il corpo del messaggio)
        if not current_text and text_lower.startswith(INTESTAZIONI_DATA):
            current_msg["data"] = text.split(":", 1)[-1].strip()
        elif not current_text and text_lower.startswith(INTESTAZIONI_AUTORE):
            current_msg["autore"] = text.split(":", 1)[-1].strip()
        else:
            # 3. Corpo del messaggio
            current_text.append(text)

    # Salva l'ultimo messaggio
    if current_msg is not None:
        current_msg["testo"] = "\n".join(current_text).strip()
        messages.append(current_msg)

    # Scrittura del CSV
    with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
        fieldnames = ["fonte", "data", "autore", "testo"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for msg in messages:
            writer.writerow({
                "fonte": msg.get("fonte", "N/D"),
                "data": msg.get("data", "N/D"),
                "autore": msg.get("autore", "N/D"),
                "testo": msg.get("testo", ""),
            })

    return len(messages)

In [19]:
import os
import pandas as pd

# I tuoi percorsi su Drive
FILE_WORD_IN = '/content/drive/MyDrive/comunicazioni_e_note.docx'
FILE_CSV_OUT = '/content/drive/MyDrive/messaggi_estratti.csv'

# 1. Esegue la funzione di estrazione (Cella 2)
try:
    numero_messaggi = extract_messages_to_csv(FILE_WORD_IN, FILE_CSV_OUT)

    print(f"✅ Estrazione completata: {numero_messaggi} messaggi.")
    print(f"📌 NUMERO DA SCRIVERE NELLA CARD: {numero_messaggi}\n")
    print("👀 Anteprima della tabella:")

    # 2. Legge il CSV appena creato e lo mostra formattato su Colab
    df_messaggi = pd.read_csv(FILE_CSV_OUT)

    # display() genera una tabella interattiva nativa di Colab
    display(df_messaggi)

except Exception as e:
    print(f"❌ ERRORE: {e}")

✅ Estrazione completata: 42 messaggi.
📌 NUMERO DA SCRIVERE NELLA CARD: 42

👀 Anteprima della tabella:


,fonte,data,autore,testo
0,EMAIL 01,09/09/2025,Marta Ferraro - Supermercato Il Girasole,Oggetto: Eccedenze reparto ortofrutta\nBuongio...
1,EMAIL 02,23/09/2025,Panificio Dolci Forme,"Oggetto: Pane invenduto della sera\nSalve, abb..."
2,EMAIL 03,06/10/2025,Elena Rossetti - Pastificio Rossetti srl,Oggetto: Bancale con errore di etichetta - pas...
3,EMAIL 04,14/10/2025,Parrocchia San Giuseppe - don Luca,Oggetto: Raccolta in parrocchia\nGentili volon...
4,EMAIL 05,28/10/2025,Logistica Supermercato Il Girasole,Oggetto: Conferma ritiro di domani\nConfermiam...
5,EMAIL 06,03/11/2025,Giorgio Marchetti,"Oggetto: Problema con yogurt\nBuonasera, ieri ..."
6,EMAIL 07,12/11/2025,Amministrazione Tecnomeccanica Bruno spa,Oggetto: Richiesta attestato di donazione\nBuo...
7,EMAIL 08,21/11/2025,Istituto Comprensivo Don Milani,Oggetto: Colletta alimentare scuola\nGentile a...
8,EMAIL 09,30/11/2025,Davide Ottone,Oggetto: Grazie!\nVolevo ringraziarvi per la s...
9,EMAIL 10,03/12/2025,Cooperativa Il Ponte - Chiara Gallo,Oggetto: Coordinamento per le famiglie dello s...
